# 面试题：Session Reset 后怎样安全恢复 Agent？

恢复不能把聊天记录原样拼回 prompt；应恢复结构化 checkpoint、artifact ID、权限、预算和权威状态版本。对未确认的写操作标记 unknown 并回读，不重发。下面用采购审批后的重启事件，比较全文恢复与最小状态恢复。

## 真实案例

采购任务在审批后重启，六个 checkpoint 事件包括已验证预算、审批、草稿、未知提交、旧聊天和过期缓存。

## 基线

基线相信旧聊天中的“已提交”。

## 结果解读

手写恢复器只消费已验证 artifact，并对未知写操作回读。

## 失败案例

超时提交不能据模型文本重复下单。

In [1]:
events = [{'id':'S1','kind':'budget','verified':True,'version':3,'value':'800'}, {'id':'S2','kind':'approval','verified':True,'version':2,'value':'ticket-A'}, {'id':'S3','kind':'draft','verified':True,'version':5,'value':'PO-7'}, {'id':'S4','kind':'submit','verified':False,'version':None,'value':'timeout'}, {'id':'S5','kind':'chat_claim','verified':False,'version':None,'value':'已提交'}, {'id':'S6','kind':'cache','verified':False,'version':1,'value':'旧库存'}]  # 构造六条 session 中断前的结构化与非结构化事件。
print('Checkpoint 输入:', events)  # 输出恢复前可获得的持久化记录。
print('教学说明：verified 只由工具后置验证写入，模型文本永不等同于 checkpoint。')  # 明确可信边界。

Checkpoint 输入: [{'id': 'S1', 'kind': 'budget', 'verified': True, 'version': 3, 'value': '800'}, {'id': 'S2', 'kind': 'approval', 'verified': True, 'version': 2, 'value': 'ticket-A'}, {'id': 'S3', 'kind': 'draft', 'verified': True, 'version': 5, 'value': 'PO-7'}, {'id': 'S4', 'kind': 'submit', 'verified': False, 'version': None, 'value': 'timeout'}, {'id': 'S5', 'kind': 'chat_claim', 'verified': False, 'version': None, 'value': '已提交'}, {'id': 'S6', 'kind': 'cache', 'verified': False, 'version': 1, 'value': '旧库存'}]
教学说明：verified 只由工具后置验证写入，模型文本永不等同于 checkpoint。


In [2]:
chat_text = '预算已确认，审批通过，采购单已提交'  # 模拟旧对话中模型生成的完成性文本。
baseline = '继续通知员工' if '已提交' in chat_text else '检查状态'  # 构造全文恢复时依赖文本关键词的错误基线。
print('聊天恢复基线:', baseline)  # 输出会跳过权威回读的危险决策。

聊天恢复基线: 继续通知员工


In [3]:
def restore(records):  # 定义 session reset 后的最小状态恢复逻辑。
    verified = {row['kind']:row for row in records if row['verified']}  # 仅恢复有后置验证证据的 artifact。
    unknown_writes = [row for row in records if row['kind'] == 'submit' and not row['verified']]  # 找到重启时状态未知的写操作。
    action = 'readback_submit' if unknown_writes else 'resume_next_node'  # 未确认写操作优先进入权威回读。
    context = {'budget':verified.get('budget'),'approval':verified.get('approval'),'draft':verified.get('draft')}  # 构建不含旧聊天和过期缓存的最小 context。
    return action, context, unknown_writes  # 返回下一步、可安全上下文和未知写清单。

In [4]:
action, context, unknown = restore(events)  # 对六条事件执行结构化恢复。
print('恢复动作:', action)  # 输出重启后首个确定性控制动作。
print('最小 context:', context)  # 输出保留的已验证 artifact。
print('未知写操作:', unknown)  # 输出需要带幂等键和资源 ID 回读的提交。
print('结果解读：旧聊天不进入事实层，提交状态未知时不继续通知也不重复提交。')  # 解释 session reset 与纯 replay 的区别。

恢复动作: readback_submit
最小 context: {'budget': {'id': 'S1', 'kind': 'budget', 'verified': True, 'version': 3, 'value': '800'}, 'approval': {'id': 'S2', 'kind': 'approval', 'verified': True, 'version': 2, 'value': 'ticket-A'}, 'draft': {'id': 'S3', 'kind': 'draft', 'verified': True, 'version': 5, 'value': 'PO-7'}}
未知写操作: [{'id': 'S4', 'kind': 'submit', 'verified': False, 'version': None, 'value': 'timeout'}]
结果解读：旧聊天不进入事实层，提交状态未知时不继续通知也不重复提交。


In [5]:
wrong = baseline  # 保留全文聊天恢复的错误动作。
fixed = action  # 读取结构化恢复的安全动作。
print('失败案例：聊天=', wrong, '，checkpoint=', fixed)  # 展示恢复策略差异。
print('生产差距：需持久化任务版本、ACL、artifact hash、幂等键、检查点 TTL、恢复次数与隔离工作空间。')  # 说明长程运行控制面。

失败案例：聊天= 继续通知员工 ，checkpoint= readback_submit
生产差距：需持久化任务版本、ACL、artifact hash、幂等键、检查点 TTL、恢复次数与隔离工作空间。


In [6]:
assert action == 'readback_submit'  # 验证未知写操作会优先回读。
assert context['budget']['value'] == '800'  # 验证已验证预算被恢复。
assert 'chat_claim' not in context  # 验证模型聊天声明不会进入事实 context。